In [1]:
import pandas as pd
import networkx as nx
from pathlib import Path

from src.utils.data_loader import ROOT, load_raw_data

In [2]:
df_notas = pd.read_csv(ROOT / "data" / "processed" / "toxicidade_perspective_COMPLETO.csv")
df_notas

,id,perspective_toxicity,severe_toxicity,identity_attack,insult,profanity,threat
0,1j3g0q5,0.020607,0.000787,0.004995,0.011579,0.009733,0.006337
1,1jdfbt3,0.322899,0.017417,0.277596,0.130648,0.103639,0.035113
2,1k7jcux,0.022610,0.001249,0.006364,0.011123,0.014737,0.007405
3,1ilwij6,0.152692,0.005722,0.102216,0.033130,0.038405,0.012959
4,1jfzx63,0.172851,0.010834,0.020179,0.032028,0.076762,0.096230
...,...,...,...,...,...,...,...
1665366,14ifnii,0.054779,0.002708,0.005772,0.017872,0.024678,0.010318
1665367,zqyg79,0.038284,0.003643,0.008140,0.015341,0.027411,0.010085
1665368,c6e90e,0.432304,0.023077,0.017566,0.191015,0.454489,0.013632
1665369,r8kbmz,0.557396,0.336188,0.218660,0.466256,0.536542,0.061090


In [8]:
df_notas.isna().sum()

id                          0
perspective_toxicity    22968
severe_toxicity         22968
identity_attack         22968
insult                  22968
profanity               22968
threat                  22968
dtype: int64

In [11]:
GRAPH_PATH = ROOT / "artifacts/graph/network_disparity.graphml"
#CORPUS_PATH = ROOT / "artifacts/graph/corpus_filtered_for_nlp.parquet"

# Substitua pelo caminho real do seu arquivo de toxicidade
PATH_TOXICITY = ROOT / "data" / "processed" / "toxicidade_perspective_COMPLETO.csv" # ou .csv

def analyze_toxic_communities():
    print('='*50)
    print("1. Carregando Grafo e Mapeando Comunidades (Louvain)...")
    G = nx.read_graphml(GRAPH_PATH)
    
    # Rodamos o Louvain com seed fixa para garantir as mesmas comunidades da sua imagem
    communities = nx.community.louvain_communities(G, weight='peso_jaccard', resolution=0.5, seed=42)
    
    # Criando o dicionário {subreddit: id_comunidade}
    sub2comm = {}
    for comm_id, comm in enumerate(communities):
        for sub in comm:
            sub2comm[sub] = comm_id
            
    print(f"Total de {len(communities)} comunidades mapeadas.")

    print("\n2. Carregando Corpus Dinamicamente...")
    # Aqui usamos o only_valid_ids=True para pegar APENAS os posts com texto válido (> 1 token)
    # Trazemos apenas 'id' e 'subreddit' para economizar RAM, pois a toxicidade já está no outro arquivo
    df_textos = load_raw_data(columns=['id', 'subreddit'], only_valid_ids=True)
    
    # Filtro crucial: manter apenas os subreddits que sobreviveram na topologia (Grafo)
    subreddits_validos = list(sub2comm.keys())
    df_corpus = df_textos[df_textos['subreddit'].isin(subreddits_validos)].copy()
    print(f"Posts retidos no ecossistema (backbone): {len(df_corpus)}")

    print("\n3. Carregando Dados de Toxicidade e Cruzando...")
    df_tox = pd.read_csv(PATH_TOXICITY) 
    
    # Adiciona o ID da comunidade ao corpus
    df_corpus['community'] = df_corpus['subreddit'].map(sub2comm).astype(int)

    # Junta o corpus (com a comunidade) aos scores de toxicidade usando o 'id' do post
    df_merged = df_corpus.merge(df_tox, on='id', how='inner')
    print(f"Posts cruzados com os scores do Google Perspective com sucesso: {len(df_merged)}")

    print("\n4. Agregando Estatísticas por Comunidade...")
    # Agrupa pela comunidade e calcula a média de cada tipo de toxicidade
    comm_stats = df_merged.groupby('community').agg(
        num_posts=('id', 'count'),
        num_subs=('subreddit', 'nunique'),
        avg_toxicity=('perspective_toxicity', 'mean'),
        avg_severe=('severe_toxicity', 'mean'),
        avg_identity=('identity_attack', 'mean'),
        avg_insult=('insult', 'mean'),
        avg_threat=('threat', 'mean'),
        avg_profanity=('profanity', 'mean')
    ).reset_index()

    # Descobrir qual é o subreddit principal ("cara" da comunidade)
    top_subs = df_merged.groupby(['community', 'subreddit']).size().reset_index(name='count')
    top_subs = top_subs.sort_values('count', ascending=False).drop_duplicates('community')
    
    comm_stats = comm_stats.merge(top_subs[['community', 'subreddit']], on='community', how='left')
    comm_stats.rename(columns={'subreddit': 'top_subreddit'}, inplace=True)

    # Filtro de relevância: Ignorar comunidades minúsculas (ex: com menos de 100 posts totais) 
    comm_stats_validas = comm_stats[comm_stats['num_posts'] >= 100].copy()

    # Ordena pelas comunidades com maior ataque de identidade/toxicidade geral
    ranking = comm_stats_validas.sort_values('avg_toxicity', ascending=False)

    print("\n" + "="*50)
    print("TOP 5 COMUNIDADES MAIS TÓXICAS")
    print("="*50)
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    # Exibir o top 5 formatado
    top_5 = ranking.head(5)[['community', 'top_subreddit', 'num_subs', 'num_posts', 'avg_toxicity', 'avg_identity', 'avg_threat']]
    print(top_5.to_string(index=False))
    
    # Salvar o ranking completo
    output_path = ROOT / "reports/graph_analysis/ranking_toxicidade_comunidades.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    ranking.to_csv(output_path, index=False)
    print(f"\nRanking completo salvo em: {output_path}")

if __name__ == "__main__":
    analyze_toxic_communities()

1. Carregando Grafo e Mapeando Comunidades (Louvain)...
Total de 479 comunidades mapeadas.

2. Carregando Corpus Dinamicamente...
Posts carregados: 1665371
Quantidade de subreddts: 7341
Posts retidos no ecossistema (backbone): 1109867

3. Carregando Dados de Toxicidade e Cruzando...
Posts cruzados com os scores do Google Perspective com sucesso: 1109867

4. Agregando Estatísticas por Comunidade...

TOP 5 COMUNIDADES MAIS TÓXICAS
 community     top_subreddit  num_subs  num_posts  avg_toxicity  avg_identity  avg_threat
       362             sissy         2        583      0.630151      0.219407    0.094822
       351   gonewildstories         4       1192      0.594703      0.184598    0.105077
       122 gaysexconfessions         2        592      0.570156      0.194160    0.113586
       383      prostateplay         2        573      0.523683      0.107038    0.067830
       415        theredpill         2        588      0.466389      0.193996    0.090311

Ranking completo salvo em: